## **Respaldo completo del Servidor**

Para hacer un respaldo completo del estado actual.

La mejor estrategia es guardar dos niveles:

1. Backup lógico completo y restaurable: Ubuntu, contenedores Docker, configuraciones, volúmenes y bases de datos.
2. Imagen física del SSD con Clonezilla: copia exacta del disco, recomendable después de preparar uno de los HDD.

Como ahora tengo un solo disco, hare el primero


### **El respaldo contendrá:**
```bash
/etc: red, Cloudflare, firewall, servicios y tareas programadas.
/var/www: las cuatro instalaciones actuales de PeerTube.
/var/lib/docker: imágenes, volúmenes y estado de Docker.
/home/admin_n/nombre_contenedores: contenedores de otros proyectos.
/home/admin_n: scripts y archivos administrativos.
/usr/local/bin: scripts personalizados.
/var/backups: backups existentes.
/boot: arranque del sistema.
volcados SQL independientes de todas las bases PostgreSQL.
inventario de contenedores, imágenes, redes y volúmenes.
```
Durante la copia detendremos temporalmente los contenedores y Cloudflare para evitar que PostgreSQL o Docker cambien archivos mientras se leen. 


**Dónde se guardará**

Se creará temporalmente en:

            /home/admin_n/backups/

Después debes descargarlo a tu notebook con WinSCP. Mientras siga en el mismo SSD, todavía no cuenta como respaldo contra fallo físico del disco.

### **Crear el script de backup**
 
Para ejecutar debemos estar en: /home/admin

```bash
cd /home/admin_n

cat > backup_completo_servidor.sh <<'EOF'
#!/usr/bin/env bash

set -Eeuo pipefail

FECHA="$(date +%Y%m%d_%H%M%S)"
BASE="/home/admin_n/backups"
DESTINO="${BASE}/servidor_antes_reestructuracion_${FECHA}"
ARCHIVO="${BASE}/servidor_antes_reestructuracion_${FECHA}.tar.gz"
CONTENEDORES_ACTIVOS="${DESTINO}/contenedores_activos.ids"
CLOUDFLARED_ESTABA_ACTIVO=0
SERVICIOS_RESTAURADOS=0

restaurar_servicios() {
    if [ "$SERVICIOS_RESTAURADOS" -eq 1 ]; then
        return
    fi

    echo
    echo "Restaurando los servicios..."

    if [ -s "$CONTENEDORES_ACTIVOS" ]; then
        xargs -r docker start < "$CONTENEDORES_ACTIVOS"
    fi

    if [ "$CLOUDFLARED_ESTABA_ACTIVO" -eq 1 ]; then
        sudo systemctl start cloudflared
    fi

    SERVICIOS_RESTAURADOS=1
}

trap restaurar_servicios EXIT

mkdir -p "$DESTINO/sql"

echo "============================================================"
echo "1. REGISTRANDO EL ESTADO ACTUAL"
echo "============================================================"

docker ps -q > "$CONTENEDORES_ACTIVOS"
docker ps -a > "$DESTINO/docker-ps-a.txt"
docker compose ls --all > "$DESTINO/docker-compose-ls.txt"
docker image ls > "$DESTINO/docker-images.txt"
docker volume ls > "$DESTINO/docker-volumes.txt"
docker network ls > "$DESTINO/docker-networks.txt"
docker system df -v > "$DESTINO/docker-system-df.txt"
lsblk -f > "$DESTINO/lsblk-f.txt"
df -hT > "$DESTINO/df-hT.txt"
sudo pvs > "$DESTINO/pvs.txt"
sudo vgs > "$DESTINO/vgs.txt"
sudo lvs > "$DESTINO/lvs.txt"
sudo ss -lntup > "$DESTINO/puertos.txt"
sudo ufw status verbose > "$DESTINO/ufw.txt" || true

echo
echo "============================================================"
echo "2. CREANDO VOLCADOS DE POSTGRESQL"
echo "============================================================"

mapfile -t POSTGRES_CONTAINERS < <(
    docker ps \
        --filter "ancestor=postgres:17-alpine" \
        --format '{{.Names}}'
)

for CONTENEDOR in "${POSTGRES_CONTAINERS[@]}"; do
    USUARIO="$(
        docker inspect \
            --format '{{range .Config.Env}}{{println .}}{{end}}' \
            "$CONTENEDOR" |
        sed -n 's/^POSTGRES_USER=//p' |
        head -n 1
    )"

    if [ -z "$USUARIO" ]; then
        USUARIO="postgres"
    fi

    echo "Respaldando $CONTENEDOR..."

    docker exec "$CONTENEDOR" \
        pg_dumpall -U "$USUARIO" |
        gzip -9 > "${DESTINO}/sql/${CONTENEDOR}.sql.gz"

    if [ ! -s "${DESTINO}/sql/${CONTENEDOR}.sql.gz" ]; then
        echo "ERROR: el respaldo SQL de $CONTENEDOR está vacío."
        exit 1
    fi
done

echo
echo "============================================================"
echo "3. DETENIENDO TEMPORALMENTE LOS SERVICIOS"
echo "============================================================"

if systemctl is-active --quiet cloudflared; then
    CLOUDFLARED_ESTABA_ACTIVO=1
    sudo systemctl stop cloudflared
fi

if [ -s "$CONTENEDORES_ACTIVOS" ]; then
    xargs -r docker stop --time 60 < "$CONTENEDORES_ACTIVOS"
fi

echo
echo "============================================================"
echo "4. CREANDO ARCHIVO COMPLETO DEL SERVIDOR"
echo "============================================================"

sudo tar \
    --acls \
    --xattrs \
    --numeric-owner \
    --ignore-failed-read \
    -czpf "$ARCHIVO" \
    --exclude="$BASE" \
    --exclude=/proc \
    --exclude=/proc/\* \
    --exclude=/sys \
    --exclude=/sys/\* \
    --exclude=/dev \
    --exclude=/dev/\* \
    --exclude=/run \
    --exclude=/run/\* \
    --exclude=/tmp \
    --exclude=/tmp/\* \
    --exclude=/mnt \
    --exclude=/mnt/\* \
    --exclude=/media \
    --exclude=/media/\* \
    --exclude=/lost+found \
    /

echo
echo "============================================================"
echo "5. RESTAURANDO LOS SERVICIOS"
echo "============================================================"

restaurar_servicios

echo
echo "============================================================"
echo "6. GENERANDO HASH DE INTEGRIDAD"
echo "============================================================"

sha256sum "$ARCHIVO" > "${ARCHIVO}.sha256"

echo
echo "============================================================"
echo "7. VERIFICANDO EL ARCHIVO"
echo "============================================================"

gzip -t "$ARCHIVO"
sudo tar -tzf "$ARCHIVO" >/dev/null

echo
echo "============================================================"
echo "BACKUP COMPLETADO"
echo "============================================================"
echo
echo "Archivo principal:"
echo "$ARCHIVO"
echo
echo "Hash SHA-256:"
echo "${ARCHIVO}.sha256"
echo
echo "Volcados SQL e inventario:"
echo "$DESTINO"
echo
echo "Descarga los tres elementos a tu notebook con WinSCP."
EOF

chmod 700 /home/admin_n/backup_completo_servidor.sh
```